# ORIENT'IA — Entraînement et évaluation (ML-2 à ML-6, ML-8)

Livrable 7 du sujet. Comme le notebook d'exploration, celui-ci **n'implémente rien** : il
appelle `src/ml/` et commente. Le code qui compte est testé
(`backend/tests/ml/`) et rejouable en une commande :

```bash
cd backend && python -m tests.eval_ml    # produit tests/eval_results_ml.json
```

**Ce que le sujet exige et où c'est traité :** stratégie de séparation (§2), modèle de
référence (§3), comparaison d'au moins deux approches (§4), métriques adaptées — « une
simple valeur d'accuracy ne constitue pas une évaluation suffisante » (§5), analyse des
erreurs et des biais (§6), intégration comme outil appelable (§8).

In [1]:
# On remonte a backend/ pour que `src` soit importable (cf. pyproject.toml).
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))

In [2]:
from src.ml.donnees_synthetiques import charger_jeu_de_donnees
from src.ml.entrainement import (
    entrainer_baseline,
    entrainer_baseline_calibree,
    entrainer_foret,
    preparer_jeu_entrainement,
    separer_indices,
    separer_train_test,
)
from src.ml.evaluation import (
    analyser_erreurs,
    evaluer_chemin_de_production,
    evaluer_modele,
    matrice_confusion,
    mesurer_stabilite,
    mesurer_stabilite_des_recommandations,
)

exemples = charger_jeu_de_donnees()
X, y = preparer_jeu_entrainement(exemples)
X_train, X_test, y_train, y_test = separer_train_test(X, y)
print(f'X={X.shape}  train={len(y_train)}  test={len(y_test)}')

X=(800, 156)  train=600  test=200


## 1. Variables et stratégie de séparation (ML-2)

**Variables.** Encodage multi-hot sur un vocabulaire contrôlé dérivé des archétypes, plus
une note par matière et un one-hot sur l'environnement recherché (`ml/features.py`).
L'entrée est *ouverte* — « maths », « Python », « SVT » sont ramenés au vocabulaire par
`ml/vocabulaire.py` — mais l'espace de features reste fermé, puisque c'est dessus que le
modèle est entraîné.

**Séparation.** Split stratifié 75/25 : chaque parcours doit être présent dans le test,
sans quoi les métriques par classe seraient incalculables pour lui.

**Ce que cette séparation ne mesure pas.** Le montage recommandé par le §5 du sujet est
« synthétique pour l'entraînement, enquête réelle pour la validation et le test ». Faute
d'enquête (DATA-4, bloquée hors hackathon), train et test viennent tous deux du
générateur : les chiffres ci-dessous mesurent la capacité du modèle à **retrouver les
hypothèses de génération**, pas à orienter de vrais candidats. C'est la limite ML-7, et
aucune métrique de ce notebook ne la contourne.

## 2. Deux approches comparées (ML-3, ML-4)

- **Baseline** : régression logistique multinomiale — linéaire, rapide, et surtout
  interprétable par classe (`coef_`), ce qui alimente les justifications de
  `identifier_points_forts`.
- **Comparaison** : forêt aléatoire — capture des interactions qu'un modèle linéaire ne
  représente pas.

Pas de LightGBM : sur quelques centaines de profils, la forêt donne une seconde approche
suffisamment différente sans ajouter de dépendance de boosting à l'installation.

In [3]:
baseline = entrainer_baseline(X_train, y_train)
production = entrainer_baseline_calibree(X_train, y_train)  # le modele SERVI
foret = entrainer_foret(X_train, y_train)

r_baseline = evaluer_modele(baseline, X_test, y_test)
r_production = evaluer_modele(production, X_test, y_test)
r_foret = evaluer_modele(foret, X_test, y_test)

cles = ['exactitude', 'f1_macro', 'top_3_accuracy', 'mrr', 'ndcg_3', 'pr_auc_macro']
print(f"{'metrique':22} {'brut':>10} {'calibre':>10} {'foret':>10}")
for cle in cles:
    print(f'{cle:22} {r_baseline[cle]:>10.4f} {r_production[cle]:>10.4f} '
          f'{r_foret[cle]:>10.4f}')

metrique                     brut    calibre      foret
exactitude                 0.9950     0.9950     0.8600
f1_macro                   0.9948     0.9950     0.8602
top_3_accuracy             1.0000     1.0000     0.9950
mrr                        0.9975     0.9975     0.9254
ndcg_3                     0.9982     0.9982     0.9424
pr_auc_macro               0.9996     1.0000     0.9661


**Le choix suit la mesure, pas l'inverse.** La régression logistique généralise mieux ici
que la forêt, qui surapprend davantage sur un jeu de cette taille avec ce niveau de bruit.
C'est pourquoi `ml/outils.py` sert la régression logistique en production. La forêt reste
le second modèle exigé par le sujet, et redeviendra un candidat sérieux si le jeu grossit
(enquête réelle) ou si des interactions s'avèrent importantes.

## 3. Métriques adaptées (ML-5)

Le §7 est explicite : « une simple valeur d'accuracy ne constitue pas une évaluation
suffisante ». Le système propose des parcours **ordonnés** : les métriques de classement
sont donc les métriques pertinentes, pas l'exactitude au rang 1.

In [4]:
print('--- classement (modele servi) ---')
for cle in ['top_3_accuracy', 'mrr', 'ndcg_3', 'rang_median_bonne_classe']:
    print(f'  {cle:28} {r_production[cle]}')

print('\n--- calibration : avant / apres ---')
brut, cal = r_baseline['calibration'], r_production['calibration']
print(f"  ECE brut                         {brut['ece']:.4f}")
print(f"  ECE calibre (servi)              {cal['ece']:.4f}")
print(f"  Brier brut / calibre             {brut['score_de_brier']:.4f}"
      f" / {cal['score_de_brier']:.4f}")

# Le SENS de l'ecart, que l'ECE (valeur absolue) ne donne pas.
for nom, c in (('brut', brut), ('calibre', cal)):
    signe = c['ecart_signe_confiance_moins_exactitude']
    sens = 'SUR-confiance' if signe > 0 else 'SOUS-confiance'
    print(f"  ecart signe {nom:8} {signe:+.4f}  ({sens})")

--- classement (modele servi) ---
  top_3_accuracy               1.0
  mrr                          0.9975
  ndcg_3                       0.9981546487678572
  rang_median_bonne_classe     1.0

--- calibration : avant / apres ---
  ECE brut                         0.1197
  ECE calibre (servi)              0.0334
  Brier brut / calibre             0.0336 / 0.0163
  ecart signe brut     -0.1163  (SOUS-confiance)
  ecart signe calibre  -0.0325  (SOUS-confiance)


**Pourquoi l'ECE et non la seule séparation de confiance.** La séparation (confiance
moyenne quand le modèle a raison vs quand il a tort) dit si la confiance *discrimine* ;
elle ne dit pas si elle est *juste*. Un modèle qui annonce 90 % et réussit 60 % du temps
est discriminant et pourtant trompeur — et c'est un score d'adéquation affiché à un
candidat. L'ECE mesure précisément cet écart, par tranche de confiance.

In [5]:
print('calibration par tranche de confiance :')
print(f"{'intervalle':>16} {'effectif':>9} {'confiance':>10} {'exactitude':>11}")
for t in cal['tranches']:
    print(f"{str(t['intervalle']):>16} {t['effectif']:>9} "
          f"{t['confiance_moyenne']:>10.3f} {t['exactitude']:>11.3f}")

calibration par tranche de confiance :
      intervalle  effectif  confiance  exactitude
      [0.4, 0.5]         1      0.420       1.000
      [0.5, 0.6]         2      0.545       0.500
      [0.6, 0.7]         3      0.663       1.000
      [0.7, 0.8]         8      0.768       1.000
      [0.8, 0.9]         7      0.862       1.000
      [0.9, 1.0]       179      0.988       1.000


## 4. Analyse des erreurs (ML-6)

In [6]:
print('confusions les plus frequentes (foret) :')
for e in analyser_erreurs(foret, X_test, y_test):
    print(f"  {e['vrai']:10} -> {e['predit']:10} x{e['occurrences']}")

confusion = matrice_confusion(baseline, X_test, y_test)
hors_diagonale = {
    vraie: {p: n for p, n in predites.items() if p != vraie}
    for vraie, predites in confusion['par_vraie_classe'].items()
}
print('\nmatrice de confusion, cases hors diagonale (baseline) :')
for vraie, predites in hors_diagonale.items():
    if predites:
        print(f'  {vraie:10} -> {predites}')

confusions les plus frequentes (foret) :
  IAA        -> DTJA       x2
  EMP        -> DTJA       x2
  IGGLIA     -> EMP        x2
  TEH        -> PIP        x2
  IAA        -> CAA        x1

matrice de confusion, cases hors diagonale (baseline) :
  IMTICIA    -> {np.str_('ICMP'): 1}


**Lecture attendue.** Les confusions doivent porter sur des parcours **proches**
(même mention, archétypes voisins). Une confusion entre deux parcours sans rapport
signalerait un problème du générateur ou du modèle, pas une ambiguïté légitime.

## 5. Stabilité — deux notions distinctes

- `mesurer_stabilite` : variance de l'exactitude entre découpages train/test. Une propriété
  de l'**entraînement**.
- `mesurer_stabilite_des_recommandations` : la recommandation résiste-t-elle au retrait
  d'un trait déclaré ? C'est ce que le §7 nomme « stabilité des recommandations », et c'est
  la propriété qui compte pour un candidat : un assistant dont le parcours de tête bascule
  parce qu'on a retiré un centre d'intérêt sur cinq n'est pas utilisable, quelle que soit
  son exactitude.

In [7]:
print('stabilite d entrainement (5 graines) :')
s = mesurer_stabilite(entrainer_foret, X, y)
print(f"  exactitude moyenne {s['exactitude_moyenne']:.4f}  "
      f"ecart-type {s['exactitude_ecart_type']:.4f}")

stabilite d entrainement (5 graines) :


  exactitude moyenne 0.8580  ecart-type 0.0242


## 6. Le chemin réellement servi (ML-8, §8 du sujet)

`evaluer_modele` note un estimateur scikit-learn sur des vecteurs. **Ce n'est pas ce que
l'assistant exécute.** Entre les deux s'intercalent la résolution du vocabulaire ouvert, le
garde-fou d'exploitabilité et les règles d'admission du volet hybride, qui *réordonnent* le
classement. Mesurer le seul estimateur reviendrait à publier les chiffres d'un modèle que
personne n'exécute — ce que le §8 interdit et que le §14 contrôle.

Le modèle entraîné sur le seul train est imposé aux outils le temps de la mesure, sans quoi
les profils de test auraient été vus à l'entraînement.

In [8]:
from src.ml import outils
from src.schemas import ProfilCandidat

_, indices_test = separer_indices(y)
exemples_test = [
    {
        'profil': ProfilCandidat.model_validate(exemples[i]['profil']),
        'parcours_id': exemples[i]['parcours_id'],
    }
    for i in indices_test
]

outils.imposer_modele_pour_evaluation(baseline)
try:
    production = evaluer_chemin_de_production(exemples_test, outils.analyser_profil)
    stabilite = mesurer_stabilite_des_recommandations(exemples_test, outils.analyser_profil)
finally:
    outils.imposer_modele_pour_evaluation(None)

print('chemin de production (analyser_profil) :')
for cle, valeur in production.items():
    print(f'  {cle:34} {valeur}')
print('\nstabilite des recommandations :')
for cle, valeur in stabilite.items():
    print(f'  {cle:34} {valeur}')

chemin de production (analyser_profil) :
  effectif                           200
  top_1                              0.995
  top_3                              1.0
  mrr                                0.9975
  ndcg_3                             0.9981546487678572
  rang_median_bonne_classe           1.0
  profils_juges_inexploitables       0
  parcours_absents_du_classement     0

stabilite des recommandations :
  profils_compares                   200
  perturbation                       retrait d'un trait déclaré au hasard
  top_1_inchange                     0.99
  selection_presentee_inchangee      0.935
  top_3_fixe_inchange                0.665


**Deux lectures à ne pas confondre.**

1. Le classement du chemin de production coïncide ici avec celui de l'estimateur : les
   profils synthétiques ne déclarent **aucune série de baccalauréat**, donc les règles
   d'admission ne s'appliquent à aucun d'eux. Elles ne sont pas sans effet — leur effet est
   démontré dans `backend/tests/ml/test_hybride.py` — elles sont *inapplicables à ce jeu*.
   L'écart deviendra mesurable avec l'enquête réelle (DATA-4, ML-7).
2. La stabilité du top-3 est nettement plus basse que celle du top-1 : le parcours de tête
   tient, la *composition* des trois premiers beaucoup moins. Pour un assistant qui présente
   plusieurs pistes, c'est la limite la plus concrète mesurée ici, et elle est à afficher
   comme telle plutôt qu'à masquer derrière l'exactitude.

## 7. Biais et limites (ML-6)

- **Le jeu est synthétique.** Un modèle entraîné et évalué dessus mesure sa capacité à
  retrouver les archétypes, pas à orienter (§5 du sujet). Tout chiffre de ce notebook est à
  lire sous cette réserve.
- **Les archétypes sont une hypothèse manuelle**, dérivée des descriptions officielles de
  parcours, non validée auprès d'étudiants réels.
- **Une fuite a déjà été trouvée et corrigée** (`environnement_travail_recherche`), et un
  contrôle automatique la surveille désormais (notebook 01, section 4).
- **`activites_projets` et `serie_bac` ne sont jamais générés** : deux champs listés par le
  §5 que le modèle ne peut pas apprendre en l'état.
- **Aucune variable sensible** (genre, âge, origine) n'entre dans l'espace de features —
  vérifiable dans `noms_features()`, et c'est ce qui permet à l'assistant d'affirmer que sa
  recommandation n'en dépend pas (§16).

In [9]:
from src.ml.features import noms_features

sensibles = ['genre', 'sexe', 'age', 'origine', 'religion', 'handicap', 'nationalite']
trouves = [n for n in noms_features() if any(s in n.lower() for s in sensibles)]
print(f'dimensions totales : {len(noms_features())}')
print(f'dimensions sensibles : {trouves if trouves else "aucune"}')

dimensions totales : 156
dimensions sensibles : ['interet:voyage']
